# Prueba del worker afinado en Colab

Activa un entorno con GPU y ejecuta las celdas en orden. Cuando se abra el selector de archivos, sube `worker-entrenamiento-colab.zip`, `yolo11n-colab-8fe0c3cd.pt` y un video `.mp4`.

El mismo video se procesa como grabación y como directo simulado, con el detector a 960 y la calibración automática. La prueba descarga un ZIP con los tiempos, las cajas, las posiciones del plano y capturas. El directo simulado sigue el reloj del archivo y descarta los cuadros atrasados. La medición del retraso de la cámara al navegador se realiza con el sistema desplegado.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

import torch
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError('Selecciona un acelerador GPU y ejecuta la celda de nuevo.')
print('GPU:', torch.cuda.get_device_name(0))
print('Sube el ZIP del worker, los pesos .pt y un MP4.')
subidos = files.upload()
rutas = [Path(nombre).resolve() for nombre in subidos]
subidos.clear()
paquetes = [p for p in rutas if p.name == 'worker-entrenamiento-colab.zip']
modelos = [p for p in rutas if p.suffix.lower() == '.pt']
videos = [p for p in rutas if p.suffix.lower() == '.mp4']
if len(rutas) != 3 or not (len(paquetes) == len(modelos) == len(videos) == 1):
    raise ValueError('Selecciona worker-entrenamiento-colab.zip, un .pt afinado y un .mp4.')
pesos, video = modelos[0], videos[0]
raiz = Path(mkdtemp(prefix='area-tecnica-ensayo-', dir='/content')).resolve()
with ZipFile(paquetes[0]) as paquete:
    if any(not (raiz / n).resolve().is_relative_to(raiz) for n in paquete.namelist()):
        raise ValueError('El ZIP contiene una ruta fuera de su destino.')
    paquete.extractall(raiz)
vision = raiz / 'vision'
manifiesto = json.loads((raiz / 'manifiesto.json').read_text(encoding='utf-8'))
for nombre, esperado in manifiesto.items():
    ruta = (vision / nombre).resolve()
    if not ruta.is_relative_to(vision) or hashlib.sha256(ruta.read_bytes()).hexdigest() != esperado:
        raise ValueError(f'El archivo {nombre} no coincide con el ZIP del worker.')
sys.path.insert(0, str(vision))
print(f'Worker verificado ({len(manifiesto)} archivos). Video: {video.name}. Pesos: {pesos.name}.')

In [ ]:
import subprocess

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r',
                str(vision / 'requirements.txt'), 'ultralytics==8.4.215'], check=True)
from calibracion.instalar_modelo import instalar

print('Modelo de calibración verificado:', instalar())

In [ ]:
import gc

from deteccion.modelo import CLASES_PROPIO, ConfigModelo, ModeloDeteccion
from evaluacion.ensayo_video import ensayar

SEGUNDOS = 60
TAMANO_IMG = 960
COLOR_LOCAL = '#C8102E'  # Cambia los colores según las camisetas de tu clip.
COLOR_VISITANTE = '#17202A'
calibracion = dict(modo='AUTOMATICA', partido_id=1, largo_m=105, ancho_m=68)
salida = Path(mkdtemp(prefix='resultado-worker-', dir=raiz))
comparacion = {}
for nombre, directo in (('grabacion', False), ('directo-simulado', True)):
    modelo = ModeloDeteccion(ConfigModelo(ruta_pesos=str(pesos), clases=dict(CLASES_PROPIO),
                                         dispositivo='cuda:0', tamano_img=TAMANO_IMG))
    print(f'Procesando {nombre} a {TAMANO_IMG}', flush=True)
    carpeta = salida / nombre
    resumen = ensayar(modelo, video, carpeta, segundos=SEGUNDOS, simular_directo=directo,
                      calibracion=calibracion, color_propio=COLOR_LOCAL, color_rival=COLOR_VISITANTE)
    if not str(resumen['dispositivo']).startswith('cuda'):
        raise RuntimeError(f'El modo {nombre} no ejecutó la inferencia en GPU.')
    muestras = json.loads((carpeta / 'muestras.json').read_text(encoding='utf-8'))
    tiempos = [m['t'] for m in muestras if m['calidad']['calibracion_valida']]
    saltos = [b - a for a, b in zip(tiempos, tiempos[1:])]
    resumen.update(
        gpu=torch.cuda.get_device_name(0),
        primer_t_con_calibracion_valida_s=tiempos[0] if tiempos else None,
        mayor_salto_entre_muestras_validas_s=round(max(saltos), 3) if saltos else None,
        muestras_con_balon_en_el_plano=sum(m['balon'] is not None for m in muestras),
    )
    (carpeta / 'resumen.json').write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding='utf-8')
    comparacion[nombre] = resumen
    print(json.dumps(resumen, ensure_ascii=False, indent=2))
    del modelo
    gc.collect()
    torch.cuda.empty_cache()
(salida / 'comparacion.json').write_text(json.dumps(comparacion, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
import shutil

paquete_resultados = shutil.make_archive(str(salida), 'zip', root_dir=salida)
files.download(paquete_resultados)
print('ZIP de resultados:', paquete_resultados)